In [1]:
import argparse
import logging
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")

In [2]:
# Stations with monthly measurements
MONTHLY_STATIONS = [16, 23, 26, 27, 28, 29, 35]

# Stations with quarterly measurements (~4 measurements per year).
# Station 25 changes from monthly to quarterly frequency from 2009 onward.
QUARTERLY_STATIONS = [14, 15, 17, 19, 20, 21, 22, 24, 25, 30, 31, 32, 33, 34]

# Allowed NH4 threshold in Ukraine (mg/dm3)
THRESHOLD = 0.5

# Values above this limit are considered measurement errors and removed
HARD_MAX = 100.0

# Valid date range for the dataset
DATE_MIN, DATE_MAX = "1990-01-01", "2025-12-31"

# Robust z-score threshold for outlier detection.
# Computed on log1p-transformed, seasonally adjusted values.
OUTLIER_K = 5.0

# Action applied to detected outliers:
# "winsorize" = cap extreme values
# "remove"    = remove outliers
# "flag"      = only mark outliers
OUTLIER_ACTION = "winsorize"

# Minimum number of observations required to calculate a station-month median
MIN_GROUP = 8

# Start year for yearly backtesting; the model is refitted each year
FIRST_TEST_YEAR = 2014

# Regularization strength for the Ridge model
RIDGE_ALPHA = 5.0

# Prediction interval confidence levels
LEVELS = (0.80, 0.95)

# All models evaluated in the experiment
MODELS_ALL = [
    "persistence",
    "seasonal_naive",
    "climatology",
    "ridge",
    "hgb"
]

# Models that require training
MODELS_TRAINED = ["ridge", "hgb"]

# Display labels for the models
MODEL_LABEL = {
    "persistence": "Persistence",
    "seasonal_naive": "Seasonal naive",
    "climatology": "Climatology",
    "ridge": "Ridge (pooled)",
    "hgb": "HistGB (pooled)",
}

# Colors used for plotting each model
MODEL_COLOR = {
    "persistence": "#7f7f7f",
    "seasonal_naive": "#bcbd22",
    "climatology": "#1f77b4",
    "ridge": "#d62728",
    "hgb": "#2ca02c",
}

# Base features used by the models
FEATS = ["a0", "a1", "a2", "r"]

# Features used by the Gradient Boosting model,
# including seasonal and spatial features
FEATS_GB = FEATS + ["sin", "cos", "dist"]


@dataclass
class Spec:
    # Name of the temporal resolution
    name: str

    # Number of months between observations:
    # 1 = monthly, 3 = quarterly
    step: int

    # Number of seasons per year:
    # 12 for monthly data, 4 for quarterly data
    S: int

    # Maximum forecast horizon (in time steps)
    H: int

    # Rolling window size for calculating deviation
    # (measured in years)
    roll: int

    # Maximum number of consecutive missing steps
    # that can be interpolated for features
    interp_limit: int

    # Minimum number of samples required
    # to calibrate prediction intervals
    min_cal: int

    # Maximum allowed age of the latest station measurement
    # in time steps
    stale: int

    # Pandas frequency string
    freq: str

    # Time unit used in the specification
    unit: str


# Configuration for monthly data
MONTHLY = Spec(
    "monthly",
    1,      # One month per step
    12,     # 12 seasons per year
    6,      # Maximum forecast horizon: 6 months
    6,      # Rolling window: 6 years
    2,      # Interpolate at most 2 missing steps
    60,     # Minimum calibration samples
    3,      # Maximum station-data staleness
    "MS",   # Month-start frequency
    "month"
)

# Configuration for quarterly data
QUARTERLY = Spec(
    "quarterly",
    3,      # Three months per step
    4,      # 4 seasons per year
    2,      # Maximum forecast horizon: 2 quarters
    4,      # Rolling window: 4 years
    1,      # Interpolate at most 1 missing step
    40,     # Minimum calibration samples
    1,      # Maximum station-data staleness
    "QS",   # Quarter-start frequency
    "quarter"
)


# Logger used to record information and errors during execution
log = logging.getLogger("nh4")

In [3]:
def save_fig(fig, path):
    # Adjust the layout to prevent overlapping elements
    fig.tight_layout()

    # Save the figure with high resolution and a tight bounding box
    fig.savefig(path, dpi=130, bbox_inches="tight")

    # Close the figure to free memory
    plt.close(fig)


def _fmt(v, nd=3):
    # Format date/time values as YYYY-MM-DD
    if isinstance(v, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(v).strftime("%Y-%m-%d")

    # Format floating-point values with the specified number of decimals
    if isinstance(v, (float, np.floating)):
        # Return an empty string for NaN values
        if np.isnan(v):
            return ""

        return f"{v:.{nd}f}"

    # Convert other values to strings
    return str(v)


def table_png(df, path, title, heat=None, nd=3, max_rows=70, fontsize=8):
    """
    Convert a DataFrame into a PNG table.

    heat = {column_name: 'low' | 'high'}
    - 'low': lower values are better
    - 'high': higher values are better
    """

    # Keep only the first max_rows rows and reset the index
    d = df.head(max_rows).reset_index(drop=True)

    # Format all table values for display
    cells = [[_fmt(v, nd) for v in row]
             for row in d.itertuples(index=False)]

    # Get the number of columns and rows
    ncol, nrow = d.shape[1], d.shape[0]

    # Set the figure size based on the table dimensions
    fig_w = max(6, 1.25 * ncol)
    fig_h = 0.34 * (nrow + 2) + 0.8

    # Create the figure and axis
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # Hide the axis because only the table is displayed
    ax.axis("off")

    # Add the table title
    ax.set_title(title, fontsize=11, fontweight="bold", loc="left")

    # Create the table from the DataFrame
    tb = ax.table(
        cellText=cells,
        colLabels=list(d.columns),
        loc="upper center",
        cellLoc="center"
    )

    # Use a fixed font size for the table
    tb.auto_set_font_size(False)
    tb.set_fontsize(fontsize)

    # Increase the row height for better readability
    tb.scale(1, 1.25)

    # Format table cells
    for (r, c), cell in tb.get_celld().items():
        # Set a light border color
        cell.set_edgecolor("#DDDDDD")

        if r == 0:
            # Format the header row
            cell.set_facecolor("#1F4E79")
            cell.set_text_props(color="white", fontweight="bold")
        else:
            # Alternate row colors for readability
            cell.set_facecolor(
                "#F5F8FC" if r % 2 == 0 else "white"
            )

    if heat:
        # Use a red-yellow-green color map for heatmap-style cells
        cmap = plt.get_cmap("RdYlGn")

        for col, direction in heat.items():
            # Skip columns that do not exist in the DataFrame
            if col not in d.columns:
                continue

            # Get the column index
            j = list(d.columns).index(col)

            # Convert values to numeric values
            vals = pd.to_numeric(d[col], errors="coerce")

            # Find the minimum and maximum values
            lo, hi = vals.min(), vals.max()

            # Skip columns with invalid values or no variation
            if not np.isfinite(lo) or hi == lo:
                continue

            for i, v in enumerate(vals):
                # Skip missing values
                if np.isnan(v):
                    continue

                # Normalize the value to the range [0, 1]
                x = (v - lo) / (hi - lo)

                # Reverse the scale when lower values are better
                if direction == "low":
                    x = 1 - x

                # Apply the heatmap color to the cell
                tb[(i + 1, j)].set_facecolor(
                    cmap(0.15 + 0.7 * x)
                )

    # Save the table as a PNG image
    fig.savefig(path, dpi=140, bbox_inches="tight")

    # Close the figure to free memory
    plt.close(fig)


def save_excel(tables, path):
    # Create an Excel writer using openpyxl
    with pd.ExcelWriter(path, engine="openpyxl") as w:

        # Write each DataFrame to a separate worksheet
        for name, df in tables.items():

            # Skip empty or missing DataFrames
            if df is None or len(df) == 0:
                continue

            # Excel sheet names cannot exceed 31 characters
            sheet = name[:31]

            # Export the DataFrame without its index
            df.to_excel(w, sheet_name=sheet, index=False)

            # Get the worksheet object
            ws = w.sheets[sheet]

            # Freeze the header row
            ws.freeze_panes = "A2"

            # Adjust column widths based on the content
            for j, col in enumerate(df.columns, start=1):
                width = max(
                    len(str(col)),
                    *(len(_fmt(v)) for v in df[col].head(200))
                ) + 2

                # Limit the maximum column width to 40
                ws.column_dimensions[
                    ws.cell(row=1, column=j).column_letter
                ].width = min(width, 40)

            # Make the header row bold
            for c in ws[1]:
                c.font = c.font.copy(bold=True)

In [4]:
# 1. LOAD + FILTER RAW DATA
# ==========================================================================

def load_and_filter(path):
    """
    Load the raw data, standardize data types, and remove invalid records.
    Returns:
        - df: cleaned DataFrame
        - steps: cleaning log as a DataFrame
    """

    # Load the CSV file using semicolon as the default separator
    raw = pd.read_csv(path, sep=";")

    # Fall back to comma separator if the file has only one column
    if raw.shape[1] == 1:
        raw = pd.read_csv(path, sep=",")

    # Remove leading/trailing spaces from column names
    raw.columns = [c.strip() for c in raw.columns]

    # Check whether all required columns are present
    need = {"ID_Station", "Date", "NH4", "Distance"}
    if not need.issubset(raw.columns):
        raise ValueError(f"Missing columns: {need - set(raw.columns)}")

    # Store the number of rows removed at each cleaning step
    steps = []

    def add(step, n, action):
        # Record the cleaning step and write it to the log
        steps.append({
            "step": step,
            "rows_affected": int(n),
            "action": action
        })
        log.info(
            "[clean] %-42s %6d  -> %s",
            step, n, action
        )

    # Create a working copy of the raw data
    df = raw.copy()

    # Record the initial number of rows
    add("0. Rows in raw file", len(df), "-")

    # Convert station IDs to numeric values
    df["ID_Station"] = pd.to_numeric(
        df["ID_Station"],
        errors="coerce"
    )

    # Count and remove invalid station IDs
    n = df["ID_Station"].isna().sum()
    df = df.dropna(subset=["ID_Station"])
    df["ID_Station"] = df["ID_Station"].astype(int)

    add("1. Invalid station ID", n, "dropped")

    # Convert dates from DD.MM.YYYY format to datetime
    df["Date"] = pd.to_datetime(
        df["Date"],
        format="%d.%m.%Y",
        errors="coerce"
    )

    # Count and remove dates that cannot be parsed
    n = df["Date"].isna().sum()
    df = df.dropna(subset=["Date"])

    add("2. Unparsable date (dd.mm.yyyy)", n, "dropped")

    # Identify dates outside the valid date range
    m = (df["Date"] < DATE_MIN) | (df["Date"] > DATE_MAX)

    # Remove records with invalid dates
    df = df[~m]

    add("3. Date outside plausible range", m.sum(), "dropped")

    # Convert NH4 values to numeric.
    # Replace decimal commas with decimal points first.
    df["NH4"] = pd.to_numeric(
        df["NH4"].astype(str).str.replace(",", "."),
        errors="coerce"
    )

    # Count and remove missing or non-numeric NH4 values
    n = df["NH4"].isna().sum()
    df = df.dropna(subset=["NH4"])

    add("4. NH4 missing / non-numeric", n, "dropped")

    # Identify negative NH4 values, which are physically impossible
    m = df["NH4"] < 0

    # Remove negative NH4 values
    df = df[~m]

    add("5. NH4 negative (impossible)", m.sum(), "dropped")

    # Identify unrealistically large NH4 values
    m = df["NH4"] > HARD_MAX

    # Remove values above the predefined hard limit
    df = df[~m]

    add(
        f"6. NH4 > {HARD_MAX:g} mg/dm3 (instrument error)",
        m.sum(),
        "dropped"
    )

    # Count and remove completely duplicated rows
    n = df.duplicated().sum()
    df = df.drop_duplicates()

    add("7. Exact duplicate rows", n, "dropped")

    # Group measurements by station and date
    # to identify multiple NH4 values for the same station/date
    g = df.groupby(["ID_Station", "Date"])["NH4"]

    # Count records belonging to station-date groups with duplicates
    n = (g.transform("size") > 1).sum()

    # Aggregate duplicate station-date measurements:
    # use the mean NH4 and keep the first Distance value
    df = (
        df.groupby(
            ["ID_Station", "Date"],
            as_index=False
        )
        .agg(
            NH4=("NH4", "mean"),
            Distance=("Distance", "first")
        )
    )

    add(
        "8. Same station+date, different value",
        n,
        "averaged"
    )

    # Count the number of distinct Distance values for each station
    nd = df.groupby("ID_Station")["Distance"].nunique()

    # Report stations whose Distance value is not consistent
    add(
        "9. Stations with >1 distinct Distance",
        (nd > 1).sum(),
        "kept first value" if (nd > 1).any() else "none (OK)"
    )

    # Count zero NH4 measurements
    # Keep them because log1p(0) is well-defined
    add(
        "10. NH4 == 0 (below detection limit?)",
        (df["NH4"] == 0).sum(),
        "kept (log1p handles 0)"
    )

    # Create a year-month period for each measurement
    # and identify station-months with multiple samples
    n_month = (
        df.assign(ym=df["Date"].dt.to_period("M"))
        .groupby(["ID_Station", "ym"])
        .size()
        .gt(1)
        .sum()
    )

    # Multiple samples in the same month will be averaged later
    add(
        "11. Station-months with >1 sample",
        n_month,
        "averaged later at monthly aggregation"
    )

    # Sort the cleaned data by station and date
    # and reset the row index
    df = (
        df.sort_values(["ID_Station", "Date"])
        .reset_index(drop=True)
    )

    # Record the final number of rows after cleaning
    add(
        "12. Rows kept after filtering",
        len(df),
        "-"
    )

    # Return the cleaned data and the cleaning summary
    return df, pd.DataFrame(steps)

In [5]:
# ==========================================================================
# 2. OUTLIER DETECTION AND HANDLING
# ==========================================================================

def handle_outliers(df):
    """
    Detect outliers using a robust z-score on log1p(NH4),
    after removing seasonal effects by station and month.

    Large NH4 values may represent real pollution events,
    so the default action is to winsorize rather than remove them.
    """

    # Create a copy to avoid modifying the original DataFrame
    d = df.copy()

    # Apply log1p transformation to reduce the effect of extreme values
    d["lg"] = np.log1p(d["NH4"])

    # Extract the month to account for seasonal patterns
    d["month"] = d["Date"].dt.month

    # Group transformed NH4 by station and calendar month
    g = d.groupby(["ID_Station", "month"])["lg"]

    # Calculate the seasonal median and number of observations
    med_sm, n_sm = g.transform("median"), g.transform("count")

    # Calculate the overall median for each station
    # Used when the station-month group has too few observations
    med_s = d.groupby("ID_Station")["lg"].transform("median")

    # Use the station-month median when enough data are available;
    # otherwise, use the station-level median
    d["expected_lg"] = np.where(
        n_sm >= MIN_GROUP,
        med_sm,
        med_s
    )

    # Calculate the residual from the expected seasonal value
    d["res"] = d["lg"] - d["expected_lg"]

    # Calculate the Median Absolute Deviation (MAD) for each station
    # MAD is more robust to extreme values than standard deviation
    mad = d.groupby("ID_Station")["res"].transform(
        lambda x: np.median(np.abs(x - np.median(x)))
    )

    # Convert MAD to a robust scale estimate.
    # Use a minimum value to avoid division by values close to zero.
    d["scale"] = np.maximum(
        mad / 0.6745,
        0.05
    )

    # Calculate the robust z-score
    d["z"] = d["res"] / d["scale"]

    # Flag observations whose absolute z-score exceeds the threshold
    d["is_outlier"] = d["z"].abs() > OUTLIER_K

    # Calculate the upper/lower boundary used for winsorization
    cap = (
        d["expected_lg"]
        + np.sign(d["z"]) * OUTLIER_K * d["scale"]
    )

    if OUTLIER_ACTION == "winsorize":
        # Replace extreme values with the corresponding boundary
        # while keeping the observation in the dataset
        lg_clean = np.where(
            d["is_outlier"],
            cap,
            d["lg"]
        )

        # Convert values back from log scale to the original NH4 scale
        d["NH4_clean"] = np.expm1(lg_clean).clip(min=0)

    elif OUTLIER_ACTION == "remove":
        # Replace detected outliers with missing values
        d["NH4_clean"] = np.where(
            d["is_outlier"],
            np.nan,
            d["NH4"]
        )

    else:
        # Keep the original NH4 values and only flag the outliers
        d["NH4_clean"] = d["NH4"]

    # Convert the expected value back to the original NH4 scale
    d["expected_NH4"] = np.expm1(d["expected_lg"])

    # Log the number of detected outliers and the selected action
    log.info(
        "[outlier] %d / %d records flagged (|z|>%.1f), action: %s",
        d["is_outlier"].sum(),
        len(d),
        OUTLIER_K,
        OUTLIER_ACTION
    )

    # Return the DataFrame with outlier information and cleaned NH4 values
    return d


def outlier_tables(d):
    # Select only observations identified as outliers
    # and keep the most relevant information for reporting
    out = d[
        d["is_outlier"]
    ][
        [
            "ID_Station",
            "Date",
            "NH4",
            "expected_NH4",
            "z",
            "NH4_clean"
        ]
    ].copy()

    # Indicate whether the outlier is unusually high or low
    out["direction"] = np.where(
        out["z"] > 0,
        "high",
        "low"
    )

    # Sort outliers by the absolute magnitude of their z-score
    out = out.sort_values(
        "z",
        key=np.abs,
        ascending=False
    )

    # Count total observations and outliers for each station
    per = (
        d.groupby("ID_Station")
        .agg(
            n_obs=("NH4", "size"),
            n_outliers=("is_outlier", "sum")
        )
        .reset_index()
    )

    # Calculate the percentage of outliers at each station
    per["pct_outliers"] = (
        per["n_outliers"] / per["n_obs"] * 100
    )

    # Return the detailed outlier table and station-level summary
    return (
        out.reset_index(drop=True),
        per
    )

In [6]:
# ==========================================================================
# 3. STATION-LEVEL SUMMARY STATISTICS
# ==========================================================================

def station_summary(d):
    # Group observations by monitoring station
    g = d.groupby("ID_Station")

    # Calculate summary statistics for each station
    s = g.agg(
        # Station location/distance information
        Distance_km=("Distance", "first"),

        # Number of observations
        n_obs=("NH4", "size"),

        # First and last observation dates
        first=("Date", "min"),
        last=("Date", "max"),

        # Basic NH4 statistics
        mean=("NH4", "mean"),
        median=("NH4", "median"),

        # 90th percentile of NH4
        p90=("NH4", lambda x: x.quantile(0.9)),

        # Maximum observed NH4 value
        max=("NH4", "max"),

        # Percentage of observations above the NH4 threshold
        pct_over_0_5=(
            "NH4",
            lambda x: (x > THRESHOLD).mean() * 100
        ),

        # Number of zero NH4 measurements
        n_zero=(
            "NH4",
            lambda x: (x == 0).sum()
        ),

        # Number of observations flagged as outliers
        n_outliers=("is_outlier", "sum")
    )

    # Count observations for each station and year
    py = (
        d.groupby([
            "ID_Station",
            d["Date"].dt.year
        ])
        .size()
        .unstack(fill_value=0)

        # Keep only the years 2011-2018
        .reindex(
            columns=range(2011, 2019),
            fill_value=0
        )
    )

    # Use the median number of yearly observations
    # to represent the typical sampling frequency
    s["obs_per_year_2011_18"] = py.median(axis=1)

    # Classify stations based on their typical sampling frequency:
    # >= 10 observations/year  -> monthly
    # >= 3 observations/year   -> quarterly
    # otherwise                 -> sparse
    s["frequency"] = np.select(
        [
            s["obs_per_year_2011_18"] >= 10,
            s["obs_per_year_2011_18"] >= 3
        ],
        [
            "monthly",
            "quarterly"
        ],
        "sparse"
    )

    # Calculate the time span covered by observations in years
    s["span_years"] = (
        (s["last"] - s["first"]).dt.days / 365.25
    )

    # Reset the index and sort stations by distance
    return (
        s.reset_index()
        .sort_values("Distance_km")
    )

In [7]:
# ==========================================================================
# 4. REGULAR TIME SERIES CONSTRUCTION
# ==========================================================================

def build_series(d, stations, spec):
    """
    Aggregate observations to monthly or quarterly frequency.

    Returns:
        raw   : aggregated raw NH4 values used for evaluation
        clean : aggregated NH4 values after outlier handling
        L_tgt : log1p(clean), used as the training target
        L_feat: interpolated L_tgt, used only for features
    """

    # Keep only the selected monitoring stations
    x = d[d["ID_Station"].isin(stations)]

    # Convert dates to monthly or quarterly timestamps
    per = (
        x["Date"]
        .dt.to_period("M" if spec.step == 1 else "Q")
        .dt.to_timestamp()
    )

    # Aggregate raw NH4 measurements by time period and station
    raw = (
        x.groupby([per, x["ID_Station"]])["NH4"]
        .mean()
        .unstack()
    )

    # Aggregate cleaned NH4 measurements by time period and station
    cln = (
        x.groupby([per, x["ID_Station"]])["NH4_clean"]
        .mean()
        .unstack()
    )

    # Create a continuous monthly/quarterly time index
    idx = pd.date_range(
        raw.index.min(),
        raw.index.max(),
        freq=spec.freq
    )

    # Reindex both series to the same time grid and station list
    raw = raw.reindex(index=idx, columns=stations)
    cln = cln.reindex(index=idx, columns=stations)

    # Apply log1p transformation to the cleaned NH4 values
    L_tgt = np.log1p(cln)

    # Interpolate short gaps for feature construction only
    # The target values themselves are not modified
    L_feat = L_tgt.interpolate(
        limit=spec.interp_limit,
        limit_area="inside"
    )

    return raw, cln, L_tgt, L_feat


def season_of(idx, spec):
    # Return month number for monthly data
    # or quarter number for quarterly data
    return np.asarray(
        idx.month if spec.step == 1 else idx.quarter
    )


def climatology(L_tgt, cutoff, spec):
    # Use only data available up to the cutoff date
    tr = L_tgt.loc[:cutoff]

    # Calculate the mean value for each month or quarter
    clim = tr.groupby(
        season_of(tr.index, spec)
    ).mean()

    # Ensure all seasons are represented
    clim = clim.reindex(
        range(1, spec.S + 1)
    )

    # Fill missing seasonal means with the overall mean
    return clim.fillna(tr.mean())


def anomalies(L, clim, spec):
    # Match each observation with its corresponding
    # monthly or quarterly climatological value
    cl = clim.reindex(
        season_of(L.index, spec)
    )

    # Restore the original time index
    cl.index = L.index

    # Calculate the anomaly relative to the seasonal climatology
    return L - cl[L.columns]


def build_panel(
    stations,
    h,
    A_feat,
    A_tgt,
    L_feat,
    raw,
    clim,
    spec,
    dist
):
    """
    Build the supervised learning dataset.

    Each row represents:
        (station, forecast origin t, forecast horizon h)

    Features use only information available at or before t.
    The target is the anomaly at t + h.
    """

    # Use the feature time index as the forecast origin dates
    idx = A_feat.index

    # Calculate the target date for the selected forecast horizon
    tgt_dates = idx + pd.DateOffset(
        months=spec.step * h
    )

    # Get the month/quarter of each target date
    tgt_season = season_of(
        tgt_dates,
        spec
    )

    # Store data for all stations
    frames = []

    for s in stations:
        # Get anomaly features for the current station
        a = A_feat[s]

        # Build lagged and rolling features
        f = pd.DataFrame({
            # Current anomaly
            "a0": a,

            # Previous-step anomaly
            "a1": a.shift(1),

            # Two-step lagged anomaly
            "a2": a.shift(2),

            # Rolling mean of recent anomalies
            "r": a.rolling(
                spec.roll,
                min_periods=2
            ).mean(),
        }, index=idx)

        # Future anomaly used as the training target
        f["y"] = A_tgt[s].shift(-h)

        # Actual raw NH4 at the target date, used for evaluation
        f["act"] = raw[s].shift(-h)

        # Indicator showing whether the target observation exists
        f["obs0"] = raw[s].notna()

        # Persistence baseline:
        # use the latest available feature value as the prediction
        f["pers"] = np.expm1(L_feat[s])

        # Seasonal naive baseline:
        # use the value from the same season in the previous year
        f["snaive"] = raw[s].shift(spec.S - h)

        # Seasonal climatology for the target period
        f["clim_log"] = clim[s].reindex(
            tgt_season
        ).values

        # Encode seasonality using sine and cosine
        f["sin"] = np.sin(
            2 * np.pi * tgt_season / spec.S
        )
        f["cos"] = np.cos(
            2 * np.pi * tgt_season / spec.S
        )

        # Add station distance as a spatial feature
        f["dist"] = dist[s]

        # Store the station ID
        f["station"] = s

        # Store the forecast horizon
        f["h"] = h

        # Store the forecast origin date
        f["origin"] = idx

        # Store the target date
        f["target_date"] = tgt_dates

        # Add the station-level panel to the list
        frames.append(f)

    # Combine all stations into one training panel
    return pd.concat(
        frames,
        ignore_index=True
    )


def make_hgb():
    # Create a Histogram Gradient Boosting regression model
    return HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        min_samples_leaf=30,
        l2_regularization=1.0,
        random_state=0
    )

In [8]:
# ==========================================================================
# 5. BACKTEST
# ==========================================================================

def run_backtest(spec, L_tgt, L_feat, raw, stations, dist):
    """
    Perform rolling-origin backtesting.

    For each test year Y:
    - Train using observations whose target date is <= 31 Dec of Y-1.
    - Test on forecast origins within year Y.
    - Refit the models for each year and forecast horizon.
    """

    # Get the last year available in the dataset
    last_year = L_tgt.index.max().year

    # Store prediction results from all years and horizons
    parts = []

    # Perform backtesting year by year
    for Y in range(FIRST_TEST_YEAR, last_year + 1):

        # Training data must only use information available
        # before the beginning of the test year
        cutoff = pd.Timestamp(
            f"{Y - 1}-12-31"
        )

        # Calculate climatology using training-period data only
        clim = climatology(
            L_tgt,
            cutoff,
            spec
        )

        # Calculate seasonal anomalies for features and targets
        A_feat = anomalies(
            L_feat,
            clim,
            spec
        )

        A_tgt = anomalies(
            L_tgt,
            clim,
            spec
        )

        # Evaluate each forecast horizon separately
        for h in range(1, spec.H + 1):

            # Build the supervised learning panel
            P = build_panel(
                stations,
                h,
                A_feat,
                A_tgt,
                L_feat,
                raw,
                clim,
                spec,
                dist
            )

            # Keep rows with all required training features and targets
            ok = P[
                FEATS + ["y", "clim_log"]
            ].notna().all(axis=1)

            # Select training samples whose target date
            # is available before or at the training cutoff
            tr = P[
                ok & (P["target_date"] <= cutoff)
            ]

            # Select test samples:
            # - forecast origin belongs to year Y
            # - actual target value exists
            # - target observation exists
            # - seasonal naive baseline is available
            te = P[
                ok
                & (P["origin"].dt.year == Y)
                & P["act"].notna()
                & P["obs0"]
                & P["snaive"].notna()
            ]

            # Skip this horizon if there are too few training samples
            # or no valid test observations
            if len(tr) < 50 or te.empty:
                continue

            # Train the Ridge regression model
            rid = Ridge(
                alpha=RIDGE_ALPHA
            ).fit(
                tr[FEATS],
                tr["y"]
            )

            # Train the Histogram Gradient Boosting model
            gb = make_hgb().fit(
                tr[FEATS_GB],
                tr["y"]
            )

            # Create the output table for the current test set
            o = te[
                [
                    "station",
                    "h",
                    "origin",
                    "target_date",
                    "act"
                ]
            ].copy()

            # Persistence baseline:
            # use the latest available NH4 value
            o["persistence"] = te["pers"]

            # Seasonal naive baseline:
            # use the value from the same season in the previous year
            o["seasonal_naive"] = te["snaive"]

            # Climatology baseline:
            # convert the seasonal log-scale mean back to NH4 scale
            o["climatology"] = np.expm1(
                te["clim_log"]
            )

            # Ridge prediction:
            # predict the anomaly, add it to climatology,
            # then transform back to the original NH4 scale
            o["ridge"] = (
                np.expm1(
                    te["clim_log"]
                    + rid.predict(te[FEATS])
                )
                .clip(lower=0)
            )

            # HGB prediction using the same anomaly + climatology approach
            o["hgb"] = (
                np.expm1(
                    te["clim_log"]
                    + gb.predict(te[FEATS_GB])
                )
                .clip(lower=0)
            )

            # Store the test year
            o["year"] = Y

            # Store the number of training samples
            o["n_train"] = len(tr)

            # Add predictions from this year and horizon to the results
            parts.append(o)

    # Return an empty DataFrame if no valid backtest results exist
    if not parts:
        return pd.DataFrame()

    # Combine results from all years and horizons
    bt = pd.concat(
        parts,
        ignore_index=True
    )

    # Log a summary of the backtest
    log.info(
        "[backtest %s] %d predictions (%d stations, years %d-%d)",
        spec.name,
        len(bt),
        bt["station"].nunique(),
        bt["year"].min(),
        bt["year"].max()
    )

    # Return the complete backtest results
    return bt

In [9]:
# ==========================================================================
# 6. MODEL EVALUATION
# ==========================================================================

def metric_row(y, p):
    """
    Calculate regression metrics for one set of observations and predictions.
    """

    # Convert inputs to NumPy float arrays
    y, p = np.asarray(y, float), np.asarray(p, float)

    # Calculate prediction errors
    e = p - y

    # Calculate total squared error and total variance
    sse = (e ** 2).sum()
    sst = ((y - y.mean()) ** 2).sum()

    # Transform actual and predicted values to log scale
    ly, lp = np.log1p(y), np.log1p(p)

    # Total variance on the log scale
    sst_l = ((ly - ly.mean()) ** 2).sum()

    return {
        # Number of observations
        "n": len(y),

        # Mean Absolute Error
        "MAE": np.abs(e).mean(),

        # Root Mean Squared Error
        "RMSE": np.sqrt((e ** 2).mean()),

        # R-squared on the original scale
        "R2": 1 - sse / sst if sst > 0 else np.nan,

        # R-squared on the log-transformed scale
        "R2_log": (
            1 - ((lp - ly) ** 2).sum() / sst_l
            if sst_l > 0 else np.nan
        ),

        # Mean prediction error: positive = overprediction,
        # negative = underprediction
        "Bias": e.mean(),
    }


def metrics_by(bt, by, models=MODELS_ALL):
    """
    Calculate performance metrics for each group and model.
    """

    # Store metric results for all groups and models
    recs = []

    # Group backtest results by the requested variables
    for key, g in bt.groupby(by):

        # Convert a single grouping key into a tuple
        key = key if isinstance(key, tuple) else (key,)

        # Evaluate every model
        for m in models:
            # Calculate metrics using actual and predicted NH4 values
            r = metric_row(
                g["act"].values,
                g[m].values
            )

            # Add grouping variables and model name
            r.update(dict(zip(by, key)))
            r["model"] = m

            recs.append(r)

    # Convert results into a DataFrame
    t = pd.DataFrame(recs)

    # Extract climatology performance as the reference baseline
    ref = (
        t[t["model"] == "climatology"]
        [by + ["MAE", "RMSE"]]
        .rename(
            columns={
                "MAE": "MAE_clim",
                "RMSE": "RMSE_clim"
            }
        )
    )

    # Merge climatology metrics into the full results
    t = t.merge(
        ref,
        on=by,
        how="left"
    )

    # Calculate MAE skill relative to climatology
    # Positive values indicate lower MAE than climatology
    t["MAE_skill"] = (
        1 - t["MAE"] / t["MAE_clim"]
    )

    # Calculate RMSE skill relative to climatology
    t["RMSE_skill"] = (
        1 - t["RMSE"] / t["RMSE_clim"]
    )

    # Keep only the main output columns
    cols = (
        by
        + [
            "model",
            "n",
            "MAE",
            "RMSE",
            "R2",
            "R2_log",
            "Bias",
            "MAE_skill",
            "RMSE_skill"
        ]
    )

    return t[cols]


def exceed_by(bt, by, models=MODELS_ALL):
    """
    Evaluate how well each model detects NH4 values above the threshold.
    """

    # Store threshold-detection results
    recs = []

    # Group results by the requested variables
    for key, g in bt.groupby(by):

        # Convert a single grouping key into a tuple
        key = key if isinstance(key, tuple) else (key,)

        # True if the actual NH4 value exceeds the threshold
        a = g["act"] > THRESHOLD

        # Evaluate each model
        for m in models:

            # True if the predicted NH4 value exceeds the threshold
            p = g[m] > THRESHOLD

            # Confusion matrix components
            tp = int((p & a).sum())       # True Positive
            fp = int((p & ~a).sum())      # False Positive
            fn = int((~p & a).sum())      # False Negative
            tn = int((~p & ~a).sum())     # True Negative

            # Precision: proportion of predicted exceedances
            # that are actually exceedances
            prec = (
                tp / (tp + fp)
                if tp + fp else np.nan
            )

            # Recall: proportion of actual exceedances
            # that are correctly detected
            rec = (
                tp / (tp + fn)
                if tp + fn else np.nan
            )

            # F1 score: harmonic mean of precision and recall
            f1 = (
                2 * prec * rec / (prec + rec)
                if prec and rec and (prec + rec)
                else np.nan
            )

            # Store grouping information
            r = dict(zip(by, key))

            # Store threshold classification metrics
            r.update(
                model=m,
                n=len(g),
                base_rate=a.mean(),
                TP=tp,
                FP=fp,
                FN=fn,
                TN=tn,
                accuracy=(tp + tn) / len(g),
                precision=prec,
                recall=rec,
                F1=f1
            )

            recs.append(r)

    # Return threshold detection results as a DataFrame
    return pd.DataFrame(recs)


def pick_primary(bt):
    """
    Select the trained model with the lowest average MAE
    across forecast horizons and print a model comparison.
    """

    # Calculate performance metrics by forecast horizon
    t = metrics_by(
        bt,
        ["h"],
        MODELS_TRAINED
    )

    # Summarize the average performance of each trained model
    model_summary = (
        t.groupby("model")
        .agg(
            mean_MAE=("MAE", "mean"),
            mean_RMSE=("RMSE", "mean"),
            mean_R2=("R2", "mean"),
        )
        .reset_index()
        .sort_values("mean_MAE")
    )

    # Print the comparison so we can see why a model was selected
    print("\n" + "=" * 65)
    print("MODEL COMPARISON")
    print("=" * 65)

    print(
        model_summary.to_string(
            index=False,
            formatters={
                "mean_MAE": "{:.4f}".format,
                "mean_RMSE": "{:.4f}".format,
                "mean_R2": "{:.4f}".format,
            },
        )
    )

    # Select the model with the lowest average MAE
    primary = model_summary.iloc[0]["model"]

    print("-" * 65)
    print(f"Selected primary model: {MODEL_LABEL[primary]}")
    print("Selection criterion: lowest mean MAE across horizons")
    print("=" * 65 + "\n")

    return primary


def attach_intervals(bt, primary, min_cal):
    """
    Construct prediction intervals using historical log-scale residuals
    from previous years for the same forecast horizon.

    This ensures that interval calibration uses only past out-of-sample
    information.
    """

    # Create a copy of the backtest results
    d = bt.copy()

    # Calculate log-scale prediction residuals
    d["res_log"] = (
        np.log1p(d["act"])
        - np.log1p(d[primary])
    )

    # Initialize lower and upper interval columns
    for lv in LEVELS:
        d[f"lo{int(lv * 100)}"] = np.nan
        d[f"hi{int(lv * 100)}"] = np.nan

    # Process each forecast horizon and test year separately
    for (h, Y), g in d.groupby(["h", "year"]):

        # Use only residuals from previous years
        # for the same forecast horizon
        hist = (
            d[
                (d["h"] == h)
                & (d["year"] < Y)
            ]["res_log"]
            .dropna()
        )

        # Skip calibration if there are too few historical residuals
        if len(hist) < min_cal:
            continue

        # Calculate prediction intervals for each confidence level
        for lv in LEVELS:

            # Convert confidence level to two-sided tail probability
            a = (1 - lv) / 2

            # Get lower and upper residual quantiles
            ql, qh = hist.quantile(
                [a, 1 - a]
            )

            # Convert primary model predictions to log scale
            base = np.log1p(
                g[primary]
            )

            # Calculate lower prediction bound
            d.loc[
                g.index,
                f"lo{int(lv * 100)}"
            ] = np.expm1(
                base + ql
            ).clip(lower=0)

            # Calculate upper prediction bound
            d.loc[
                g.index,
                f"hi{int(lv * 100)}"
            ] = np.expm1(
                base + qh
            )

    # Store interval coverage results
    recs = []

    # Evaluate interval performance for each forecast horizon
    for h, g in d.groupby("h"):

        # Evaluate each nominal confidence level
        for lv in LEVELS:

            lo = f"lo{int(lv * 100)}"
            hi = f"hi{int(lv * 100)}"

            # Keep observations with valid intervals
            gg = g.dropna(
                subset=[lo]
            )

            if gg.empty:
                continue

            # Check whether the actual value falls inside the interval
            inside = (
                (gg["act"] >= gg[lo])
                & (gg["act"] <= gg[hi])
            )

            # Store empirical coverage and average interval width
            recs.append({
                "h": h,
                "nominal": lv,
                "n": len(gg),
                "coverage": inside.mean(),
                "mean_width": (
                    gg[hi] - gg[lo]
                ).mean()
            })

    # Return predictions with intervals and interval evaluation results
    return d, pd.DataFrame(recs)

In [10]:
# ==========================================================================
# 7. FUTURE FORECAST
# ==========================================================================

def final_forecast(
    spec,
    stations,
    L_tgt,
    L_feat,
    raw,
    dist,
    primary,
    bt
):
    """
    Generate future NH4 forecasts for all stations and forecast horizons.

    The selected primary model is retrained using all available data.
    Prediction intervals are estimated from historical backtest residuals.
    """

    # Get the last available time point in the dataset
    end = L_tgt.index.max()

    # Calculate climatology using all available historical data
    clim = climatology(
        L_tgt,
        end,
        spec
    )

    # Calculate seasonal anomalies for features and targets
    A_feat = anomalies(
        L_feat,
        clim,
        spec
    )

    A_tgt = anomalies(
        L_tgt,
        clim,
        spec
    )

    # Find the most recent valid observation for each station
    last_obs = {
        s: raw[s].last_valid_index()
        for s in stations
    }

    # Find the latest observation date across all stations
    global_end = max(
        v for v in last_obs.values()
        if v is not None
    )

    # Store historical log-scale residuals by forecast horizon
    res_pool = {}

    if not bt.empty:
        # Calculate prediction residuals on the log scale
        rr = (
            np.log1p(bt["act"])
            - np.log1p(bt[primary])
        )

        # Pool residuals separately for each forecast horizon
        res_pool = {
            h: rr[
                bt["h"] == h
            ].dropna().values
            for h in range(1, spec.H + 1)
        }

    # Store forecast results and skipped stations
    rows, skipped = [], []

    # Generate forecasts for each horizon
    for h in range(1, spec.H + 1):

        # Build the supervised learning panel
        P = build_panel(
            stations,
            h,
            A_feat,
            A_tgt,
            L_feat,
            raw,
            clim,
            spec,
            dist
        )

        # Keep rows with all required features and target information
        ok = P[
            FEATS + ["y", "clim_log"]
        ].notna().all(axis=1)

        tr = P[ok]

        # Retrain the selected primary model using all available data
        if primary == "ridge":
            model = Ridge(
                alpha=RIDGE_ALPHA
            ).fit(
                tr[FEATS],
                tr["y"]
            )
        else:
            model = make_hgb().fit(
                tr[FEATS_GB],
                tr["y"]
            )

        # Select the corresponding feature set for the model
        cols = (
            FEATS
            if primary == "ridge"
            else FEATS_GB
        )

        # Generate forecasts for each station
        for s in stations:

            # Get the latest observation date for this station
            lo_ = last_obs[s]

            # Calculate how far behind the station is
            # compared with the latest observation across all stations
            gap = (
                (global_end.year - lo_.year) * 12
                + global_end.month - lo_.month
                if lo_ is not None
                else 999
            )

            # Skip stations with no recent observation
            # or observations that are too old
            if (
                lo_ is None
                or gap > spec.stale * spec.step
            ):
                if s not in [x[0] for x in skipped]:
                    skipped.append((s, lo_))
                continue

            # Get the row corresponding to the station's latest observation
            row = P[
                (P["station"] == s)
                & (P["origin"] == lo_)
            ].iloc[0]

            # Prepare model input features.
            # Missing lag features are filled with 0,
            # representing no anomaly relative to climatology.
            X = (
                row[cols]
                .astype(float)
                .fillna(0.0)
                .to_frame()
                .T
            )

            # Predict the anomaly and add it to climatology
            pl = (
                row["clim_log"]
                + float(model.predict(X)[0])
            )

            # Convert the prediction back to the original NH4 scale
            point = float(
                np.expm1(pl)
            )

            # Store the point forecast and supporting information
            rec = {
                "station": s,
                "origin": lo_,
                "h": h,
                "target_date": row["target_date"],
                "last_obs_value": raw[s].loc[lo_],
                "point": max(point, 0.0),
                "climatology": float(
                    np.expm1(row["clim_log"])
                )
            }

            # Get historical residuals for this forecast horizon
            rs = res_pool.get(
                h,
                np.array([])
            )

            # Build prediction intervals if enough residuals are available
            for lv in LEVELS:

                # Calculate the lower/upper tail probability
                a = (1 - lv) / 2

                if len(rs) >= 20:
                    # Calculate historical residual quantiles
                    ql, qh = np.quantile(
                        rs,
                        [a, 1 - a]
                    )

                    # Convert interval bounds back to NH4 scale
                    rec[
                        f"lo{int(lv * 100)}"
                    ] = max(
                        float(np.expm1(pl + ql)),
                        0.0
                    )

                    rec[
                        f"hi{int(lv * 100)}"
                    ] = float(
                        np.expm1(pl + qh)
                    )

                else:
                    # Not enough historical residuals for reliable intervals
                    rec[
                        f"lo{int(lv * 100)}"
                    ] = np.nan

                    rec[
                        f"hi{int(lv * 100)}"
                    ] = np.nan

            # Estimate the probability that future NH4
            # will exceed the threshold
            rec["p_exceed_0_5"] = (
                float(
                    np.mean(
                        np.expm1(pl + rs)
                        > THRESHOLD
                    )
                )
                if len(rs) >= 20
                else np.nan
            )

            # Add the forecast record to the result list
            rows.append(rec)

    # Convert all forecast records into a DataFrame
    fc = pd.DataFrame(rows)

    if not fc.empty:

        # Convert target dates to monthly or quarterly labels
        per = fc["target_date"].dt.to_period(
            "M" if spec.step == 1 else "Q"
        )

        # Insert a readable target-period column
        fc.insert(
            4,
            "target",
            per.astype(str)
        )

        # Store the model used for the final forecast
        fc["model"] = primary

    # Log stations that were skipped because their observations were too old
    for s, lo_ in skipped:
        log.warning(
            "[forecast %s] Station %s skipped: "
            "last observation %s < common cutoff %s",
            spec.name,
            s,
            None if lo_ is None else lo_.strftime("%Y-%m"),
            global_end.strftime("%Y-%m")
        )

    # Return the forecast table and skipped stations
    return fc, skipped

In [11]:
# ==========================================================================
# 8. VISUALIZATION
# ==========================================================================

def fig_availability(d, path):
    # Count observations for each station and year
    piv = (
        d.groupby(["ID_Station", d["Date"].dt.year])
         .size()
         .unstack(fill_value=0)
    )

    # Get station distances and order stations from upstream to downstream
    dist = d.groupby("ID_Station")["Distance"].first()
    piv = piv.loc[dist.sort_values().index]

    fig, ax = plt.subplots(figsize=(14, 7))

    # Plot the observation count as a heatmap
    im = ax.imshow(
        piv.values,
        aspect="auto",
        cmap="YlGnBu"
    )

    # Set year labels
    ax.set_xticks(range(piv.shape[1]))
    ax.set_xticklabels(
        piv.columns,
        rotation=90,
        fontsize=8
    )

    # Set station labels with distance
    ax.set_yticks(range(piv.shape[0]))
    ax.set_yticklabels(
        [f"{s} ({dist[s]:.0f} km)" for s in piv.index],
        fontsize=8
    )

    # Display observation counts inside each cell
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]

            if v:
                ax.text(
                    j,
                    i,
                    v,
                    ha="center",
                    va="center",
                    fontsize=6,
                    color=(
                        "white"
                        if v > piv.values.max() * 0.55
                        else "black"
                    )
                )

    ax.set_title(
        "Number of observations per station-year "
        "(stations ordered upstream -> downstream)"
    )

    # Add the color scale
    fig.colorbar(im, ax=ax, shrink=0.8)

    save_fig(fig, path)


def fig_river_profile(d, summ, path):
    fig, ax = plt.subplots(figsize=(12, 5))

    # Color monthly stations differently from other stations
    col = np.where(
        summ["Distance_km"].isin(
            summ[
                summ["ID_Station"].isin(MONTHLY_STATIONS)
            ]["Distance_km"]
        ),
        "#d62728",
        "#1f77b4"
    )

    # Plot mean NH4 concentration along the river
    ax.bar(
        summ["Distance_km"],
        summ["mean"],
        width=6,
        color=col
    )

    # Add the NH4 threshold line
    ax.axhline(
        THRESHOLD,
        color="k",
        ls="--",
        lw=1,
        label=f"Limit {THRESHOLD} mg/dm3"
    )

    # Use a symmetric log scale to handle small and large values
    ax.set_yscale(
        "symlog",
        linthresh=0.5
    )

    ax.set_xlabel("Distance from source (km)")
    ax.set_ylabel("Mean NH4 (mg/dm3, symlog)")

    # Label each station on the bars
    for _, r in summ.iterrows():
        ax.text(
            r["Distance_km"],
            r["mean"] * 1.05 + 0.02,
            int(r["ID_Station"]),
            ha="center",
            fontsize=7
        )

    # Plot the percentage of observations above the threshold
    ax2 = ax.twinx()
    ax2.plot(
        summ["Distance_km"],
        summ["pct_over_0_5"],
        "o-",
        color="orange",
        lw=1
    )

    ax2.set_ylabel(
        "% observations > 0.5",
        color="orange"
    )

    ax.set_title(
        "River profile: mean NH4 (bars; red = monthly stations "
        "used for forecast) and exceedance rate"
    )

    ax.legend(loc="upper left")

    save_fig(fig, path)


def fig_timeseries(rec, raw, cln, stations, path):
    # Create one subplot for each station
    n = len(stations)

    fig, axes = plt.subplots(
        n,
        1,
        figsize=(14, 2.6 * n),
        sharex=True
    )

    axes = np.atleast_1d(axes)

    for ax, s in zip(axes, stations):

        # Select records for the current station
        r = rec[
            rec["ID_Station"] == s
        ]

        # Plot raw monthly mean NH4
        ax.plot(
            raw.index,
            raw[s],
            color="#888",
            lw=0.8,
            label="Monthly mean (raw)"
        )

        # Plot monthly mean after outlier handling
        ax.plot(
            cln.index,
            cln[s],
            color="#1f77b4",
            lw=1,
            label="Monthly mean (after outlier handling)"
        )

        # Highlight observations flagged as outliers
        o = r[
            r["is_outlier"]
        ]

        ax.scatter(
            o["Date"],
            o["NH4"],
            marker="x",
            color="red",
            s=30,
            zorder=5,
            label="Flagged outlier (record)"
        )

        # Add the NH4 threshold line
        ax.axhline(
            THRESHOLD,
            color="k",
            ls="--",
            lw=0.8
        )

        ax.set_ylabel(
            f"St {s}\nmg/dm3"
        )

        # Use symmetric log scale
        ax.set_yscale(
            "symlog",
            linthresh=0.1
        )

    # Add a common legend and title
    axes[0].legend(
        ncol=3,
        fontsize=8,
        loc="upper left"
    )

    axes[0].set_title(
        "NH4 monthly time series by station "
        "(symlog y-axis; dashed = 0.5 limit)"
    )

    save_fig(fig, path)


def fig_seasonality(raw, stations, path):
    # Determine subplot layout
    n = len(stations)
    ncol = 2
    nrow = int(
        np.ceil(n / ncol)
    )

    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=(13, 2.8 * nrow),
        sharex=True
    )

    # Create one boxplot for each station
    for ax, s in zip(
        axes.ravel(),
        stations
    ):
        x = raw[s].dropna()

        # Group NH4 values by calendar month
        data = [
            x[x.index.month == m].values
            for m in range(1, 13)
        ]

        # Plot monthly distributions
        ax.boxplot(
            data,
            showfliers=False,
            patch_artist=True,
            boxprops=dict(
                facecolor="#cfe2f3"
            ),
            medianprops=dict(
                color="red"
            )
        )

        # Add the threshold line
        ax.axhline(
            THRESHOLD,
            color="k",
            ls="--",
            lw=0.8
        )

        ax.set_title(
            f"Station {s}",
            fontsize=9
        )

        ax.set_xticks(
            range(1, 13),
            labels=[
                str(m)
                for m in range(1, 13)
            ]
        )

    # Hide unused subplots
    for ax in axes.ravel()[n:]:
        ax.axis("off")

    fig.suptitle(
        "Seasonality: NH4 by calendar month "
        "(outliers hidden; dashed = 0.5)",
        y=1.0
    )

    save_fig(fig, path)


def fig_month_heat(raw, stations, path):
    # Calculate mean NH4 for each station and calendar month
    m = pd.DataFrame({
        s: raw[s].groupby(raw.index.month).mean()
        for s in stations
    }).T

    fig, ax = plt.subplots(
        figsize=(11, 3.8)
    )

    # Plot the station-month means as a heatmap
    im = ax.imshow(
        m.values,
        aspect="auto",
        cmap="YlOrRd"
    )

    ax.set_xticks(
        range(12)
    )
    ax.set_xticklabels(
        range(1, 13)
    )

    ax.set_yticks(
        range(len(m))
    )
    ax.set_yticklabels(
        m.index
    )

    # Display the mean value inside each cell
    for i in range(m.shape[0]):
        for j in range(12):
            ax.text(
                j,
                i,
                f"{m.values[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=7
            )

    ax.set_title(
        "Mean NH4 (mg/dm3) by station x calendar month"
    )

    ax.set_xlabel("Month")
    ax.set_ylabel("Station")

    # Add the color scale
    fig.colorbar(im, ax=ax)

    save_fig(fig, path)


def fig_acf_corr(
    L_tgt,
    stations,
    path_acf,
    path_corr
):
    # Calculate monthly climatology on the log scale
    clim = L_tgt.groupby(
        L_tgt.index.month
    ).mean()

    # Remove seasonal effects to obtain anomalies
    A = (
        L_tgt
        - clim.reindex(
            L_tgt.index.month
        ).set_axis(L_tgt.index)
    )

    # --------------------------------------------------
    # Plot autocorrelation
    # --------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 4.5)
    )

    for s in stations:
        # Keep valid anomaly values
        x = A[s].dropna()

        # Calculate autocorrelation for lags 1-24 months
        ac = [
            A[s].corr(
                A[s].shift(k)
            )
            for k in range(1, 25)
        ]

        ax.plot(
            range(1, 25),
            ac,
            marker="o",
            ms=3,
            label=f"St {s}"
        )

    # Add zero correlation line
    ax.axhline(
        0,
        color="k",
        lw=0.5
    )

    # Add approximate 95% significance bands
    ax.axhline(
        1.96 / np.sqrt(250),
        color="gray",
        ls=":"
    )

    ax.axhline(
        -1.96 / np.sqrt(250),
        color="gray",
        ls=":"
    )

    ax.set_xlabel(
        "Lag (months)"
    )

    ax.set_ylabel(
        "Autocorrelation"
    )

    ax.set_title(
        "ACF of deseasonalised log(1+NH4) "
        "(dotted = approx. 95% band)"
    )

    ax.legend(
        ncol=4,
        fontsize=8
    )

    save_fig(
        fig,
        path_acf
    )

    # --------------------------------------------------
    # Plot cross-station correlation
    # --------------------------------------------------

    # Calculate correlation between station anomalies
    C = A[stations].corr()

    fig, ax = plt.subplots(
        figsize=(6.5, 5.5)
    )

    # Plot the correlation matrix as a heatmap
    im = ax.imshow(
        C.values,
        cmap="RdBu_r",
        vmin=-1,
        vmax=1
    )

    ax.set_xticks(
        range(len(stations))
    )
    ax.set_yticks(
        range(len(stations))
    )

    ax.set_xticklabels(
        stations
    )
    ax.set_yticklabels(
        stations
    )

    # Display correlation values inside the matrix
    for i in range(len(stations)):
        for j in range(len(stations)):
            v = C.values[i, j]

            ax.text(
                j,
                i,
                "" if np.isnan(v) else f"{v:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

    ax.set_title(
        "Cross-station correlation of monthly anomalies (lag 0)"
    )

    fig.colorbar(
        im,
        ax=ax
    )

    save_fig(
        fig,
        path_corr
    )

    # Return the correlation matrix for further analysis
    return C


def fig_distribution(d, stations, path):
    # Select NH4 observations from the selected stations
    x = d[
        d["ID_Station"].isin(stations)
    ]["NH4"]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 3.8)
    )

    # Plot the raw NH4 distribution
    axes[0].hist(
        x,
        bins=60,
        color="#6fa8dc"
    )

    axes[0].axvline(
        THRESHOLD,
        color="k",
        ls="--"
    )

    axes[0].set_title(
        "NH4 (raw) - right-skewed"
    )

    # Plot the log-transformed distribution
    axes[1].hist(
        np.log1p(x),
        bins=60,
        color="#93c47d"
    )

    axes[1].set_title(
        "log(1 + NH4) - closer to symmetric"
    )

    save_fig(
        fig,
        path
    )


def fig_metrics_horizon(t, path, title):
    # Create one panel for each evaluation metric
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4.2)
    )

    for ax, met in zip(
        axes,
        ["MAE", "RMSE", "R2"]
    ):

        # Plot each model across forecast horizons
        for m in MODELS_ALL:
            x = (
                t[t["model"] == m]
                .sort_values("h")
            )

            ax.plot(
                x["h"],
                x[met],
                marker="o",
                color=MODEL_COLOR[m],
                label=MODEL_LABEL[m]
            )

        ax.set_xlabel(
            "Horizon"
        )

        ax.set_title(
            met
            + (
                " (mg/dm3)"
                if met != "R2"
                else ""
            )
        )

        ax.grid(
            alpha=0.3
        )

        # Add zero reference line for R2
        if met == "R2":
            ax.axhline(
                0,
                color="k",
                lw=0.6
            )

    # Add the model legend
    axes[0].legend(
        fontsize=8
    )

    fig.suptitle(
        title
    )

    save_fig(
        fig,
        path
    )


def fig_scatter(bt, primary, spec, path):
    # Select representative forecast horizons
    hs = sorted({
        1,
        max(1, spec.H // 2),
        spec.H
    })

    fig, axes = plt.subplots(
        1,
        len(hs),
        figsize=(5 * len(hs), 4.6)
    )

    axes = np.atleast_1d(
        axes
    )

    # Log-scale threshold for the scatter plot
    lim = np.log1p(
        THRESHOLD
    )

    for ax, h in zip(
        axes,
        hs
    ):
        # Select backtest results for this horizon
        g = bt[
            bt["h"] == h
        ]

        # Plot observed vs predicted values on log scale
        ax.scatter(
            np.log1p(g["act"]),
            np.log1p(g[primary]),
            s=8,
            alpha=0.4,
            color="#d62728"
        )

        # Add the ideal prediction line y = x
        mx = max(
            np.log1p(g["act"]).max(),
            np.log1p(g[primary]).max()
        )

        ax.plot(
            [0, mx],
            [0, mx],
            "k--",
            lw=1
        )

        # Mark the NH4 threshold
        ax.axvline(
            lim,
            color="gray",
            ls=":"
        )

        ax.axhline(
            lim,
            color="gray",
            ls=":"
        )

        # Calculate metrics for this horizon
        r = metric_row(
            g["act"],
            g[primary]
        )

        # Show performance metrics in the subplot title
        ax.set_title(
            f"h={h}: R2={r['R2']:.2f}, "
            f"MAE={r['MAE']:.3f}, "
            f"RMSE={r['RMSE']:.3f}",
            fontsize=9
        )

        ax.set_xlabel(
            "Observed log(1+NH4)"
        )

        ax.set_ylabel(
            "Predicted log(1+NH4)"
        )

    fig.suptitle(
        f"Observed vs predicted "
        f"({MODEL_LABEL[primary]}); "
        f"dotted = 0.5 mg/dm3"
    )

    save_fig(
        fig,
        path
    )

In [12]:
def fig_heat(t, value, path, title, cmap, fmt="{:.3f}"):
    # Reshape data into a station x horizon matrix
    p = t.pivot(
        index="station",
        columns="h",
        values=value
    )

    fig, ax = plt.subplots(
        figsize=(1.0 * p.shape[1] + 3, 0.5 * p.shape[0] + 2)
    )

    # Plot the values as a heatmap
    im = ax.imshow(
        p.values,
        aspect="auto",
        cmap=cmap
    )

    # Set horizon labels
    ax.set_xticks(range(p.shape[1]))
    ax.set_xticklabels(p.columns)

    # Set station labels
    ax.set_yticks(range(p.shape[0]))
    ax.set_yticklabels(p.index)

    # Display the value inside each cell
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            v = p.values[i, j]

            if not np.isnan(v):
                ax.text(
                    j,
                    i,
                    fmt.format(v),
                    ha="center",
                    va="center",
                    fontsize=8
                )

    ax.set_xlabel("Horizon")
    ax.set_ylabel("Station")
    ax.set_title(title)

    # Add the color scale
    fig.colorbar(im, ax=ax)

    save_fig(fig, path)


def fig_backtest_ts(bt, primary, h, stations, path):
    # Keep only stations available in the backtest results
    stations = [
        s for s in stations
        if s in set(bt["station"])
    ]

    n = len(stations)

    # Stop if there are no valid stations
    if n == 0:
        return

    fig, axes = plt.subplots(
        n,
        1,
        figsize=(14, 2.5 * n),
        sharex=True
    )

    axes = np.atleast_1d(axes)

    for ax, s in zip(axes, stations):
        # Select backtest results for this station and horizon
        g = (
            bt[
                (bt["station"] == s) &
                (bt["h"] == h)
            ]
            .sort_values("target_date")
        )

        # Plot observed values
        ax.plot(
            g["target_date"],
            g["act"],
            color="k",
            lw=1,
            marker="o",
            ms=2,
            label="Observed"
        )

        # Plot predictions from the selected model
        ax.plot(
            g["target_date"],
            g[primary],
            color=MODEL_COLOR[primary],
            lw=1,
            label=MODEL_LABEL[primary]
        )

        # Plot the climatology baseline
        ax.plot(
            g["target_date"],
            g["climatology"],
            color=MODEL_COLOR["climatology"],
            lw=0.8,
            ls="--",
            label="Climatology"
        )

        # Add the NH4 threshold
        ax.axhline(
            THRESHOLD,
            color="gray",
            ls=":"
        )

        ax.set_ylabel(f"St {s}")

    axes[0].legend(
        ncol=3,
        fontsize=8
    )

    axes[0].set_title(
        f"Backtest: observed vs forecast at horizon h={h}"
    )

    save_fig(fig, path)


def fig_error_box(bt, primary, path):
    # Get all available forecast horizons
    hs = sorted(
        bt["h"].unique()
    )

    fig, ax = plt.subplots(
        figsize=(9, 4.2)
    )

    # Calculate prediction errors for each horizon
    data = [
        (
            bt[bt["h"] == h][primary]
            - bt[bt["h"] == h]["act"]
        ).values
        for h in hs
    ]

    # Plot the error distribution for each horizon
    ax.boxplot(
        data,
        showfliers=False,
        patch_artist=True,
        boxprops=dict(
            facecolor="#f4cccc"
        ),
        medianprops=dict(
            color="k"
        )
    )

    ax.set_xticklabels(hs)

    # Add zero-error reference line
    ax.axhline(
        0,
        color="k",
        lw=0.6
    )

    ax.set_xlabel("Horizon")
    ax.set_ylabel(
        "Error = predicted - observed (mg/dm3)"
    )

    ax.set_title(
        f"Error distribution by horizon "
        f"({MODEL_LABEL[primary]}, outliers hidden)"
    )

    save_fig(fig, path)


def fig_season_error(bt, primary, spec, path):
    # Copy the backtest results to avoid modifying the original data
    b = bt.copy()

    # Determine the season of each target date
    b["season"] = season_of(
        pd.DatetimeIndex(b["target_date"]),
        spec
    )

    # Calculate MAE by season for the selected model and climatology
    g = b.groupby("season").apply(
        lambda x: pd.Series({
            primary: np.abs(
                x[primary] - x["act"]
            ).mean(),

            "climatology": np.abs(
                x["climatology"] - x["act"]
            ).mean()
        })
    )

    fig, ax = plt.subplots(
        figsize=(9, 4)
    )

    w = 0.38
    xs = np.arange(
        len(g)
    )

    # Plot climatology MAE
    ax.bar(
        xs - w / 2,
        g["climatology"],
        w,
        color=MODEL_COLOR["climatology"],
        label="Climatology"
    )

    # Plot selected model MAE
    ax.bar(
        xs + w / 2,
        g[primary],
        w,
        color=MODEL_COLOR[primary],
        label=MODEL_LABEL[primary]
    )

    ax.set_xticks(xs)
    ax.set_xticklabels(g.index)

    ax.set_xlabel(
        f"Target {spec.unit}"
    )

    ax.set_ylabel(
        "MAE (mg/dm3)"
    )

    ax.set_title(
        f"MAE by target {spec.unit}"
    )

    ax.legend()

    save_fig(fig, path)


def fig_exceed(ex, path):
    # Create one panel for each classification metric
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4)
    )

    for ax, met in zip(
        axes,
        ["precision", "recall", "F1"]
    ):

        # Plot each model across forecast horizons
        for m in MODELS_ALL:
            x = (
                ex[ex["model"] == m]
                .sort_values("h")
            )

            ax.plot(
                x["h"],
                x[met],
                marker="o",
                color=MODEL_COLOR[m],
                label=MODEL_LABEL[m]
            )

        ax.set_title(
            met + " (exceed 0.5 mg/dm3)"
        )

        ax.set_xlabel("Horizon")

        # Metrics are bounded between 0 and 1
        ax.set_ylim(0, 1)

        ax.grid(
            alpha=0.3
        )

    axes[0].legend(
        fontsize=8
    )

    save_fig(fig, path)


def fig_coverage(cov, path):
    # Skip plotting if no coverage results are available
    if cov.empty:
        return

    fig, ax = plt.subplots(
        figsize=(9, 4)
    )

    w = 0.38

    # Get all available horizons
    hs = sorted(
        cov["h"].unique()
    )

    # Plot empirical coverage for each prediction interval
    for k, lv in enumerate(LEVELS):
        x = (
            cov[cov["nominal"] == lv]
            .sort_values("h")
        )

        ax.bar(
            np.array(x["h"]) + (k - 0.5) * w,
            x["coverage"],
            w,
            label=f"{int(lv * 100)}% interval"
        )

        # Add the nominal coverage level as a reference line
        ax.hlines(
            lv,
            min(hs) - 0.6,
            max(hs) + 0.6,
            color="k",
            ls=":",
            lw=0.8
        )

    ax.set_ylim(
        0,
        1.05
    )

    ax.set_xlabel("Horizon")

    ax.set_ylabel(
        "Empirical coverage (out-of-sample)"
    )

    ax.set_title(
        "Prediction-interval coverage "
        "(dotted lines = nominal)"
    )

    ax.legend()

    save_fig(fig, path)


def fig_forecast(
    spec,
    raw,
    fc,
    stations,
    path,
    hist_steps,
    ncols=2
):
    # Keep only stations with available forecasts
    stations = (
        [
            s for s in stations
            if s in set(fc["station"])
        ]
        if not fc.empty
        else []
    )

    # Stop if there are no forecast results
    if not stations:
        return

    # Determine the subplot layout
    nrow = int(
        np.ceil(len(stations) / ncols)
    )

    fig, axes = plt.subplots(
        nrow,
        ncols,
        figsize=(7 * ncols, 3.2 * nrow)
    )

    axes = np.atleast_1d(
        axes
    ).ravel()

    for ax, s in zip(
        axes,
        stations
    ):
        # Get forecasts for the current station
        f = (
            fc[fc["station"] == s]
            .sort_values("h")
        )

        # Select the most recent historical observations
        h = (
            raw[s]
            .dropna()
            .iloc[-hist_steps:]
        )

        # Plot historical observations
        ax.plot(
            h.index,
            h.values,
            color="k",
            lw=1,
            marker="o",
            ms=3,
            label="Observed"
        )

        # Plot point forecasts
        ax.plot(
            f["target_date"],
            f["point"],
            color="#d62728",
            marker="o",
            ms=4,
            label="Forecast"
        )

        # Plot the climatology baseline
        ax.plot(
            f["target_date"],
            f["climatology"],
            color="#1f77b4",
            ls="--",
            lw=0.9,
            label="Climatology"
        )

        # Plot prediction intervals if available
        if f["lo95"].notna().any():

            # 95% prediction interval
            ax.fill_between(
                f["target_date"],
                f["lo95"],
                f["hi95"],
                color="#d62728",
                alpha=0.12,
                label="95%"
            )

            # 80% prediction interval
            ax.fill_between(
                f["target_date"],
                f["lo80"],
                f["hi80"],
                color="#d62728",
                alpha=0.25,
                label="80%"
            )

        # Add the NH4 threshold
        ax.axhline(
            THRESHOLD,
            color="gray",
            ls=":"
        )

        ax.set_title(
            f"Station {s}",
            fontsize=10
        )

    # Hide unused subplots
    for ax in axes[len(stations):]:
        ax.axis("off")

    axes[0].legend(
        fontsize=7,
        ncol=3
    )

    fig.suptitle(
        f"{spec.name.capitalize()} forecast "
        f"with 80%/95% intervals "
        f"(dotted = 0.5 mg/dm3)"
    )

    save_fig(fig, path)

In [13]:
# ==========================================================================
# 9. COMPLETE FORECASTING MODULE (MONTHLY OR QUARTERLY)
# ==========================================================================

def run_module(spec, stations, d, dist, figdir, tabdir, tables, summary):
    # Get the module name: "monthly" or "quarterly"
    tag = spec.name

    # Log the module configuration and forecast horizons
    log.info("=" * 70)
    log.info("MODULE %s | stations: %s | horizon 1..%d", tag.upper(), stations, spec.H)

    # Build regular time series:
    # raw = original NH4, cln = cleaned NH4,
    # L_tgt = log-transformed target, L_feat = interpolated features
    raw, cln, L_tgt, L_feat = build_series(d, stations, spec)

    # Run rolling-origin backtesting
    bt = run_backtest(spec, L_tgt, L_feat, raw, stations, dist)

    # Stop if there is not enough data for backtesting
    if bt.empty:
        log.warning("Not enough data for backtesting module %s", tag)
        return None

    # Select the primary model based on the lowest mean MAE
    primary = pick_primary(bt)
    summary.append(
        f"[{tag}] primary model (lowest mean MAE over horizons): {primary}"
    )

    # Add out-of-sample prediction intervals
    bt2, cov = attach_intervals(bt, primary, spec.min_cal)

    # Calculate model performance by different grouping levels
    t_h = metrics_by(bt2, ["h"])                         # by horizon
    t_sh = metrics_by(bt2, ["station", "h"])             # by station and horizon
    t_y = metrics_by(
        bt2, ["year", "h"], ["persistence", "climatology", primary]
    )                                                    # by year and horizon

    # Add season information and evaluate seasonal performance
    b3 = bt2.assign(
        season=season_of(pd.DatetimeIndex(bt2["target_date"]), spec)
    )
    t_se = metrics_by(
        b3, ["season"], ["climatology", primary]
    )

    # Evaluate performance by station
    t_s = metrics_by(
        bt2, ["station"], ["persistence", "climatology", primary]
    )

    # Evaluate threshold-exceedance classification performance
    ex_h = exceed_by(bt2, ["h"])                         # by horizon
    ex_sh = exceed_by(
        bt2, ["station"], ["climatology", primary]
    )                                                    # by station

    # --- Save tables for Excel and PNG outputs -----------------------------

    tables[f"{tag}_backtest_preds"] = bt2
    tables[f"{tag}_metrics_by_h"] = t_h
    tables[f"{tag}_metrics_station_h"] = t_sh
    tables[f"{tag}_metrics_station"] = t_s
    tables[f"{tag}_metrics_year"] = t_y
    tables[f"{tag}_metrics_season"] = t_se
    tables[f"{tag}_exceed_by_h"] = ex_h
    tables[f"{tag}_exceed_station"] = ex_sh
    tables[f"{tag}_coverage"] = cov

    # Create a wide summary table containing MAE, RMSE, and R2
    # for all models and horizons
    wide = (
        t_h.pivot(index="h", columns="model", values=["MAE", "RMSE", "R2"])
        .reindex(columns=MODELS_ALL, level=1)
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()
    tables[f"{tag}_summary_by_h"] = wide

    # Export the overall model-performance table as an image
    table_png(
        wide,
        tabdir / f"{tag}_metrics_by_horizon.png",
        f"{tag.capitalize()} backtest: MAE / RMSE / R2 by horizon and model (mg/dm3)",
        nd=3,
        fontsize=7
    )

    # Keep results for the selected primary model only
    tp = t_sh[
        t_sh["model"] == primary
    ][["station", "h", "n", "MAE", "RMSE", "R2", "MAE_skill"]]

    # Export station-level performance of the primary model
    table_png(
        tp,
        tabdir / f"{tag}_primary_station_h.png",
        f"{tag.capitalize()}: {MODEL_LABEL[primary]} by station and horizon",
        heat={
            "MAE": "low",
            "RMSE": "low",
            "R2": "high",
            "MAE_skill": "high"
        },
        max_rows=90
    )

    # Export threshold-exceedance classification metrics
    # Precision, Recall, and F1 measure how well the model detects NH4 > threshold
    table_png(
        ex_h[ex_h["model"].isin(
            ["climatology", primary, "persistence"]
        )][[
            "h", "model", "n", "base_rate",
            "TP", "FP", "FN", "TN",
            "precision", "recall", "F1"
        ]],
        tabdir / f"{tag}_exceedance.png",
        f"{tag.capitalize()}: exceedance (> {THRESHOLD}) classification",
        heat={
            "recall": "high",
            "precision": "high",
            "F1": "high"
        }
    )

    # Export prediction-interval coverage if available
    if not cov.empty:
        table_png(
            cov,
            tabdir / f"{tag}_interval_coverage.png",
            f"{tag.capitalize()}: out-of-sample interval coverage"
        )

    # --- Generate diagnostic figures ---------------------------------------

    # Plot MAE, RMSE, and R2 against forecast horizon
    fig_metrics_horizon(
        t_h,
        figdir / f"{tag}_metrics_vs_horizon.png",
        f"{tag.capitalize()} backtest {FIRST_TEST_YEAR}-{bt['year'].max()}"
    )

    # Plot observed vs predicted values
    fig_scatter(
        bt2,
        primary,
        spec,
        figdir / f"{tag}_scatter_obs_pred.png"
    )

    # Keep station-horizon results for the primary model
    tsh_p = t_sh[t_sh["model"] == primary]

    # Heatmap of MAE by station and horizon
    fig_heat(
        tsh_p,
        "MAE",
        figdir / f"{tag}_heat_mae.png",
        f"MAE by station x horizon ({MODEL_LABEL[primary]})",
        "YlOrRd"
    )

    # Heatmap of improvement over climatology
    # Positive MAE_skill means better than the climatology baseline
    fig_heat(
        tsh_p,
        "MAE_skill",
        figdir / f"{tag}_heat_skill.png",
        "MAE skill vs climatology (>0 = better than climatology)",
        "RdYlGn",
        "{:.2f}"
    )

    # Heatmap of R2 by station and forecast horizon
    fig_heat(
        tsh_p,
        "R2",
        figdir / f"{tag}_heat_r2.png",
        "R2 by station x horizon",
        "RdYlGn",
        "{:.2f}"
    )

    # Plot observed and forecast time series during backtesting
    fig_backtest_ts(
        bt2,
        primary,
        min(3, spec.H),
        stations,
        figdir / f"{tag}_backtest_timeseries.png"
    )

    # Show the distribution of prediction errors by horizon
    fig_error_box(
        bt2,
        primary,
        figdir / f"{tag}_error_by_horizon.png"
    )

    # Compare prediction errors across seasons
    fig_season_error(
        bt2,
        primary,
        spec,
        figdir / f"{tag}_error_by_season.png"
    )

    # Plot Precision, Recall, and F1 for threshold exceedance
    fig_exceed(
        ex_h,
        figdir / f"{tag}_exceed_metrics.png"
    )

    # Plot empirical coverage of the prediction intervals
    fig_coverage(
        cov,
        figdir / f"{tag}_interval_coverage.png"
    )

    # --- Generate future forecasts -----------------------------------------

    # Forecast future NH4 values using the selected primary model
    fc, skipped = final_forecast(
        spec,
        stations,
        L_tgt,
        L_feat,
        raw,
        dist,
        primary,
        bt2
    )

    # Save and visualize future forecasts if available
    if not fc.empty:
        tables[f"{tag}_forecast"] = fc

        # Select the main forecast information for display
        show = fc[[
            "station", "origin", "target",
            "point", "lo80", "hi80", "lo95", "hi95",
            "p_exceed_0_5", "climatology"
        ]].copy()

        # Convert forecast origin dates to YYYY-MM format
        show["origin"] = show["origin"].dt.strftime("%Y-%m")

        # Export forecast results as an image table
        table_png(
            show,
            tabdir / f"{tag}_forecast_table.png",
            f"{tag.capitalize()} forecast ({MODEL_LABEL[primary]}), mg/dm3",
            nd=3,
            heat={"p_exceed_0_5": "low"},
            max_rows=120,
            fontsize=7
        )

        # Plot historical values, forecasts, and prediction intervals
        fig_forecast(
            spec,
            raw,
            fc,
            stations,
            figdir / f"{tag}_forecast.png",
            hist_steps=60 if spec.step == 1 else 24
        )

    # Record stations that were skipped because their latest observation
    # was too old compared with the common reference date
    for s, lo_ in skipped:
        summary.append(
            f"[{tag}] station {s} skipped in forecast "
            f"(last obs {None if lo_ is None else lo_.strftime('%Y-%m')} too old)"
        )

    # --- Create a short text summary ---------------------------------------

    summary.append(
        f"\n[{tag}] mean metrics over the backtest by horizon:"
    )

    # Report performance of baseline models and the selected primary model
    for h in range(1, spec.H + 1):
        line = f"  h={h}: "

        for m in ["persistence", "climatology", primary]:
            r = t_h[
                (t_h["h"] == h) &
                (t_h["model"] == m)
            ].iloc[0]

            line += (
                f"{MODEL_LABEL[m]} "
                f"MAE={r['MAE']:.3f} "
                f"RMSE={r['RMSE']:.3f} "
                f"R2={r['R2']:.2f} | "
            )

        summary.append(line.rstrip(" |"))

    # Return the main results for further analysis
    return {
        "bt": bt2,
        "primary": primary,
        "fc": fc,
        "raw": raw,
        "t_h": t_h
    }

In [14]:
# MAIN
# ==========================================================================
def main():
    # Create the command-line argument parser
    ap = argparse.ArgumentParser()

    # Input data file
    ap.add_argument("--data", default="PB_1996_2019_NH4.csv")

    # Output directory
    ap.add_argument("--out", default="outputs")

    # Parse arguments.
    # args=[] means the notebook ignores Jupyter's own command-line arguments
    # and always uses the default values defined above.
    args = ap.parse_args(args=[])

    # Create the main output directory
    out = Path(args.out)

    # Create separate folders for figures and tables
    figdir, tabdir = out / "figures", out / "tables"
    figdir.mkdir(parents=True, exist_ok=True)
    tabdir.mkdir(parents=True, exist_ok=True)

    # Configure logging:
    # - show messages in the console
    # - also save all messages to run.log
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(
                out / "run.log",
                mode="w",
                encoding="utf-8"
            )
        ]
    )

    # Set the default font size for all matplotlib figures
    # and disable the default grid
    plt.rcParams.update({
        "font.size": 9,
        "axes.grid": False
    })

    # Dictionary used to store all output tables
    # Summary stores important text messages about the analysis
    tables = {}
    summary = [
        "NH4 forecasting pipeline - summary",
        "=" * 40
    ]

    # ----------------------------------------------------------------------
    # 1. LOAD AND CLEAN RAW DATA
    # ----------------------------------------------------------------------

    # Read the raw CSV file, validate the required columns,
    # convert data types, remove invalid records, and create a cleaning log
    df, clean_log = load_and_filter(args.data)

    # Store the cleaning log for Excel output
    tables["01_cleaning_log"] = clean_log

    # Save the cleaning log as a PNG table
    table_png(
        clean_log,
        tabdir / "01_cleaning_log.png",
        "Raw-data cleaning log",
        nd=0
    )

    # ----------------------------------------------------------------------
    # 2. OUTLIER DETECTION AND HANDLING
    # ----------------------------------------------------------------------

    # Detect potential outliers and create the cleaned NH4 values
    d = handle_outliers(df)

    # Create:
    # - a list of individual outlier records
    # - an outlier summary by station
    out_list, out_per = outlier_tables(d)

    # Store outlier results for Excel output
    tables["02_outliers_list"] = out_list
    tables["02_outliers_by_station"] = out_per

    # Save the outlier summary as a PNG table
    table_png(
        out_per.round(2),
        tabdir / "02_outliers_by_station.png",
        f"Outliers flagged (|robust z| > {OUTLIER_K}, "
        f"action = {OUTLIER_ACTION})",
        heat={"pct_outliers": "low"}
    )

    # Save the 40 records with the largest absolute outlier scores
    table_png(
        out_list.head(40),
        tabdir / "02_outliers_top40.png",
        "Top-40 flagged records",
        nd=2
    )

    # ----------------------------------------------------------------------
    # 3. STATION-LEVEL SUMMARY
    # ----------------------------------------------------------------------

    # Calculate basic statistics and observation frequency for each station
    summ = station_summary(d)

    # Store the station summary for Excel output
    tables["03_station_summary"] = summ

    # Make a copy for visualization
    ss = summ.copy()

    # Convert the first and last observation dates to YYYY-MM format
    for c in ("first", "last"):
        ss[c] = ss[c].dt.strftime("%Y-%m")

    # Save the station summary as a PNG table
    table_png(
        ss.round(2),
        tabdir / "03_station_summary.png",
        "Station summary (all stations)",
        heat={"pct_over_0_5": "low"},
        fontsize=7
    )

    # Get the actual observation frequency of each station
    cls = summ.set_index("ID_Station")["frequency"]

    # Check whether stations selected for monthly forecasting
    # actually have sufficient monthly observations
    for s in MONTHLY_STATIONS:
        if cls.get(s) != "monthly":
            log.warning(
                "Station %s was selected as a MONTHLY station, "
                "but its actual frequency is %s",
                s,
                cls.get(s)
            )

    # Check whether a station classified as monthly
    # was incorrectly placed in the quarterly station list
    for s in QUARTERLY_STATIONS:
        if cls.get(s) == "monthly":
            log.warning(
                "Station %s has monthly observations "
                "but is included in the quarterly list",
                s
            )

    # Create a dictionary:
    # station ID -> distance from the river source
    dist = (
        d.groupby("ID_Station")["Distance"]
        .first()
        .to_dict()
    )

    # ----------------------------------------------------------------------
    # 4. EXPLORATORY DATA ANALYSIS (EDA)
    # ----------------------------------------------------------------------

    # Plot the number of observations for each station and year
    fig_availability(
        d,
        figdir / "01_availability_heatmap.png"
    )

    # Plot the spatial profile of mean NH4 concentration
    # and the percentage of observations above the threshold
    fig_river_profile(
        d,
        summ,
        figdir / "02_river_profile.png"
    )

    # Plot the raw NH4 distribution for the monthly forecasting stations
    fig_distribution(
        d,
        MONTHLY_STATIONS,
        figdir / "03_distribution.png"
    )

    # Build monthly time series for the selected monthly stations
    # raw_m = raw monthly NH4
    # cln_m = monthly NH4 after outlier handling
    # Lt_m = log-transformed target
    raw_m, cln_m, Lt_m, _ = build_series(
        d,
        MONTHLY_STATIONS,
        MONTHLY
    )

    # Keep the original records belonging to the monthly stations
    rec_m = d[
        d["ID_Station"].isin(MONTHLY_STATIONS)
    ]

    # Plot monthly NH4 time series before and after outlier handling
    fig_timeseries(
        rec_m,
        raw_m,
        cln_m,
        MONTHLY_STATIONS,
        figdir / "04_timeseries_monthly.png"
    )

    # Plot seasonal distributions of NH4 by calendar month
    fig_seasonality(
        raw_m,
        MONTHLY_STATIONS,
        figdir / "05_seasonality_boxplots.png"
    )

    # Plot the mean NH4 concentration for each station and month
    fig_month_heat(
        raw_m,
        MONTHLY_STATIONS,
        figdir / "06_month_heatmap.png"
    )

    # Calculate and plot:
    # - autocorrelation of deseasonalised NH4
    # - cross-station correlation
    C = fig_acf_corr(
        Lt_m,
        MONTHLY_STATIONS,
        figdir / "07_acf.png",
        figdir / "08_corr_heatmap.png"
    )

    # Calculate mean NH4 for each station and calendar month
    mh = (
        pd.DataFrame({
            s: raw_m[s].groupby(raw_m.index.month).mean()
            for s in MONTHLY_STATIONS
        })
        .T
        .round(3)
        .reset_index()
        .rename(columns={"index": "station"})
    )

    # Store monthly means and station correlations for Excel output
    tables["04_month_means"] = mh

    tables["04_corr_anomalies"] = (
        C.round(3)
        .reset_index()
        .rename(columns={"index": "station"})
    )

    # Save monthly means as a PNG table
    table_png(
        mh,
        tabdir / "04_month_means.png",
        "Mean NH4 by station x month (mg/dm3)",
        fontsize=7
    )

    # ----------------------------------------------------------------------
    # 5-8. RUN MONTHLY AND QUARTERLY FORECASTING MODULES
    # ----------------------------------------------------------------------

    # Run the complete monthly forecasting pipeline:
    # - backtesting
    # - model comparison
    # - model selection
    # - prediction intervals
    # - future forecasting
    # - tables and figures
    res_m = run_module(
        MONTHLY,
        MONTHLY_STATIONS,
        d,
        dist,
        figdir,
        tabdir,
        tables,
        summary
    )

    # Run the same complete pipeline for quarterly forecasting
    res_q = run_module(
        QUARTERLY,
        QUARTERLY_STATIONS,
        d,
        dist,
        figdir,
        tabdir,
        tables,
        summary
    )

    # ----------------------------------------------------------------------
    # 9. EXPORT FINAL RESULTS
    # ----------------------------------------------------------------------

    # Save all stored tables into one Excel workbook
    save_excel(
        tables,
        out / "results.xlsx"
    )

    # Save the text summary
    (out / "summary.txt").write_text(
        "\n".join(summary),
        encoding="utf-8"
    )

    # Print the summary to the log
    log.info(
        "\n" + "\n".join(summary)
    )

    # Report the location of the output directory
    log.info(
        "Finished. Results saved in: %s",
        out.resolve()
    )


# Run main() only when this Python file is executed directly
# This prevents main() from running automatically when the file is imported
if __name__ == "__main__":
    main()

2026-09-24 23:42:33,293 [clean] 0. Rows in raw file                          3499  -> -
2026-09-24 23:42:33,295 [clean] 1. Invalid station ID                           0  -> dropped
2026-09-24 23:42:33,307 [clean] 2. Unparsable date (dd.mm.yyyy)                 0  -> dropped
2026-09-24 23:42:33,309 [clean] 3. Date outside plausible range                 0  -> dropped
2026-09-24 23:42:33,316 [clean] 4. NH4 missing / non-numeric                    4  -> dropped
2026-09-24 23:42:33,318 [clean] 5. NH4 negative (impossible)                    0  -> dropped
2026-09-24 23:42:33,319 [clean] 6. NH4 > 100 mg/dm3 (instrument error)          0  -> dropped
2026-09-24 23:42:33,321 [clean] 7. Exact duplicate rows                         0  -> dropped
2026-09-24 23:42:33,328 [clean] 8. Same station+date, different value           0  -> averaged
2026-09-24 23:42:33,330 [clean] 9. Stations with >1 distinct Distance           0  -> none (OK)
2026-09-24 23:42:33,330 [clean] 10. NH4 == 0 (below detection l


MODEL COMPARISON
model mean_MAE mean_RMSE mean_R2
ridge   0.2343    0.3991  0.2560
  hgb   0.2412    0.4201  0.1757
-----------------------------------------------------------------
Selected primary model: Ridge (pooled)
Selection criterion: lowest mean MAE across horizons



2026-09-24 23:42:51,267 [forecast monthly] Station 26 skipped: last observation 2018-12 < common cutoff 2019-10
2026-09-24 23:42:53,151 ======================================================================
2026-09-24 23:42:53,151 MODULE QUARTERLY | stations: [14, 15, 17, 19, 20, 21, 22, 24, 25, 30, 31, 32, 33, 34] | horizon 1..2
2026-09-24 23:42:54,818 [backtest quarterly] 478 predictions (13 stations, years 2014-2018)



MODEL COMPARISON
model mean_MAE mean_RMSE mean_R2
ridge   1.2510    3.5832  0.4773
  hgb   1.3158    3.8045  0.4110
-----------------------------------------------------------------
Selected primary model: Ridge (pooled)
Selection criterion: lowest mean MAE across horizons



2026-09-24 23:43:03,884 
NH4 forecasting pipeline - summary
[monthly] primary model (lowest mean MAE over horizons): ridge
[monthly] station 26 skipped in forecast (last obs 2018-12 too old)

[monthly] mean metrics over the backtest by horizon:
  h=1: Persistence MAE=0.224 RMSE=0.459 R2=0.02 | Climatology MAE=0.272 RMSE=0.409 R2=0.22 | Ridge (pooled) MAE=0.217 RMSE=0.376 R2=0.34
  h=2: Persistence MAE=0.282 RMSE=0.539 R2=-0.41 | Climatology MAE=0.268 RMSE=0.402 R2=0.21 | Ridge (pooled) MAE=0.233 RMSE=0.396 R2=0.24
  h=3: Persistence MAE=0.312 RMSE=0.587 R2=-0.61 | Climatology MAE=0.268 RMSE=0.407 R2=0.23 | Ridge (pooled) MAE=0.236 RMSE=0.403 R2=0.24
  h=4: Persistence MAE=0.323 RMSE=0.610 R2=-0.73 | Climatology MAE=0.268 RMSE=0.405 R2=0.24 | Ridge (pooled) MAE=0.239 RMSE=0.404 R2=0.24
  h=5: Persistence MAE=0.313 RMSE=0.584 R2=-0.57 | Climatology MAE=0.272 RMSE=0.411 R2=0.22 | Ridge (pooled) MAE=0.240 RMSE=0.407 R2=0.24
  h=6: Persistence MAE=0.319 RMSE=0.607 R2=-0.68 | Climatology MAE